# ColumnTransformer — Scikit-Learn
### Organizing Multi-Type Preprocessing in Machine Learning

Real-world datasets contain a mixture of data types: numbers, unordered categories, ordered rankings, and missing values. 
Because different columns require different transformations, we cannot simply apply one scaler or encoder to the entire table.

In this notebook, we learn how **`ColumnTransformer`** acts as a central traffic controller, directing each column group to its appropriate transformer and combining the results automatically.

## 1. Understanding the Problem

Imagine you receive the following dataset of employees. Let us inspect the columns and their data types.

In [1]:
import numpy as np
import pandas as pd

# Create a realistic sample dataset
df = pd.DataFrame({
    'Age': [25, 30, 28, 35],
    'Gender': ['Male', 'Female', 'Male', 'Female'],
    'City': ['Delhi', 'Mumbai', 'Delhi', 'Bangalore'],
    'Education': ['Graduate', 'Postgraduate', 'High School', 'Postgraduate'],
    'Salary': [50000, 70000, 40000, 85000]
})

print("--- Raw Dataset ---")
display(df)

print("\n--- Column Data Types ---")
print(df.dtypes)

--- Raw Dataset ---


,Age,Gender,City,Education,Salary
0,25,Male,Delhi,Graduate,50000
1,30,Female,Mumbai,Postgraduate,70000
2,28,Male,Delhi,High School,40000
3,35,Female,Bangalore,Postgraduate,85000



--- Column Data Types ---
Age           int64
Gender       object
City         object
Education    object
Salary        int64
dtype: object


### Interpretation
Look at the different requirements for each column:
- **`Age` & `Salary`:** Continuous numbers on completely different scales $\longrightarrow$ Need **Standardization** (`StandardScaler`).
- **`Education`:** Ordered category (`High School` < `Graduate` < `Postgraduate`) $\longrightarrow$ Needs **Ordinal Encoding** (`OrdinalEncoder`).
- **`Gender` & `City`:** Unordered nominal categories $\longrightarrow$ Need **One-Hot Encoding** (`OneHotEncoder`).

**The Problem:** Without a unified tool, you would have to manually slice columns, run each transformer separately, and stitch the arrays back together with `np.hstack()`. This manual process is lengthy, messy, and prone to column misalignment errors.

## 2. What is ColumnTransformer?

> **Formal Definition:**  
> **`ColumnTransformer`** is a Scikit-Learn utility that allows different preprocessing transformations to be applied to different subsets of columns in a single, unified step.

Let's import `ColumnTransformer` from `sklearn.compose`.

In [2]:
from sklearn.compose import ColumnTransformer

print("Successfully imported ColumnTransformer from sklearn.compose!")

Successfully imported ColumnTransformer from sklearn.compose!


### Interpretation
`ColumnTransformer` lives in `sklearn.compose` because its purpose is **composition**—combining multiple transformers into a single composite transformer.

## 3. Basic Syntax

The structure of `ColumnTransformer` is:

```python
transformer = ColumnTransformer(
    transformers=[
        ('name1', transformer1, columns1),
        ('name2', transformer2, columns2),
        ('name3', transformer3, columns3)
    ]
)
```

Each transformation is specified as a tuple containing **3 parts**:
1. **Name:** A descriptive string of your choice.
2. **Transformer:** The Scikit-Learn estimator (e.g. `OneHotEncoder()`, `StandardScaler()`).
3. **Columns:** A list of column names to receive that transformation.

In [3]:
# Conceptual demonstration of the syntax structure
from sklearn.preprocessing import OneHotEncoder

sample_transformer_tuple = ('onehot', OneHotEncoder(), ['Gender', 'City'])

print("Sample transformer step tuple:")
print("1. Step Name:       ", sample_transformer_tuple[0])
print("2. Transformer Object:", sample_transformer_tuple[1])
print("3. Target Columns:   ", sample_transformer_tuple[2])

Sample transformer step tuple:
1. Step Name:        onehot
2. Transformer Object: OneHotEncoder()
3. Target Columns:    ['Gender', 'City']


### Interpretation
This tuple tells Scikit-Learn: *"Apply `OneHotEncoder()` to `Gender` and `City`, and name this step `'onehot'`."*

## 4. Understanding (Name, Transformer, Columns)

| Part | Type | Role | Example |
| :--- | :--- | :--- | :--- |
| **Name** | `str` | Identifies the transformation step | `'ordinal_edu'` |
| **Transformer** | Estimator object | The operation to perform | `OrdinalEncoder()` |
| **Columns** | `list` of strings/ints | The columns receiving the transformation | `['Education']` |

In [4]:
# Summary of common transformers used inside ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

print("Available building blocks:")
print("- Missing Values:  SimpleImputer(strategy='median')")
print("- Numerical Scale: StandardScaler(), MinMaxScaler()")
print("- Nominal Strings: OneHotEncoder(sparse_output=False)")
print("- Ordinal Strings: OrdinalEncoder(categories=[...])")

Available building blocks:
- Missing Values:  SimpleImputer(strategy='median')
- Numerical Scale: StandardScaler(), MinMaxScaler()
- Nominal Strings: OneHotEncoder(sparse_output=False)
- Ordinal Strings: OrdinalEncoder(categories=[...])


### Interpretation
You can mix and match any Scikit-Learn transformer inside `ColumnTransformer`. It serves as the orchestrator.

## 5. Simple OneHot + Ordinal Example

Let's build a simple `ColumnTransformer` that:
1. Applies **`OrdinalEncoder`** to `Education` (with explicit order: High School < Graduate < Postgraduate).
2. Applies **`OneHotEncoder`** to `Gender` and `City`.

In [5]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# Define the ColumnTransformer
ct_simple = ColumnTransformer(
    transformers=[
        # Step 1: Ordinal encoding on Education
        (
            'ord_edu',
            OrdinalEncoder(categories=[['High School', 'Graduate', 'Postgraduate']]),
            ['Education']
        ),
        # Step 2: One-Hot encoding on Gender and City
        (
            'ohe_nominal',
            OneHotEncoder(sparse_output=False),
            ['Gender', 'City']
        )
    ]
)

print("ColumnTransformer configured:")
print(ct_simple)

ColumnTransformer configured:
ColumnTransformer(transformers=[('ord_edu',
                                 OrdinalEncoder(categories=[['High School',
                                                             'Graduate',
                                                             'Postgraduate']]),
                                 ['Education']),
                                ('ohe_nominal',
                                 OneHotEncoder(sparse_output=False),
                                 ['Gender', 'City'])])


### Interpretation
Notice that before running any data through it, our plan is completely clear and declarative. We explicitly defined the hierarchy for education and set `sparse_output=False` so we can easily view the resulting numbers.

## 6. What Happens When We Call `fit_transform()`?

When we call `fit_transform()`, `ColumnTransformer`:
1. Slices out the `Education` column and learns its categories.
2. Slices out `Gender` and `City` and learns their unique categories.
3. Transforms each slice into numbers.
4. Glues the transformed arrays side-by-side into a single 2D array.

In [6]:
# Apply the ColumnTransformer to our DataFrame
transformed_simple = ct_simple.fit_transform(df)

print("Transformed Array Shape:", transformed_simple.shape)
print("\nTransformed Array Content:")
print(transformed_simple)

Transformed Array Shape: (4, 6)

Transformed Array Content:
[[1. 0. 1. 0. 1. 0.]
 [2. 1. 0. 0. 0. 1.]
 [0. 0. 1. 0. 1. 0.]
 [2. 1. 0. 1. 0. 0.]]


### Interpretation
Notice what happened:
- Column 0 is the encoded `Education` (`1.0` for Graduate, `2.0` for Postgraduate, `0.0` for High School).
- The remaining columns are binary one-hot values for `Gender` and `City`.
- We did not have to write a single line of manual array stacking!

## 7. Understanding `remainder='drop'`

In our previous example, what happened to `Age` and `Salary`?
They were not mentioned in `transformers`.

By default, `ColumnTransformer` sets **`remainder='drop'`**, which automatically removes any column that was not assigned to a transformer.

In [7]:
# Explicitly specifying remainder='drop' (the default behavior)
ct_drop = ColumnTransformer(
    transformers=[
        ('ohe_cat', OneHotEncoder(sparse_output=False), ['Gender', 'City'])
    ],
    remainder='drop'  # Unmentioned columns (Age, Education, Salary) will be dropped
)

transformed_dropped = ct_drop.fit_transform(df)

print("Shape with remainder='drop':", transformed_dropped.shape)
print("\nTransformed Content (Only Gender & City kept):")
print(transformed_dropped)

Shape with remainder='drop': (4, 5)

Transformed Content (Only Gender & City kept):
[[0. 1. 0. 1. 0.]
 [1. 0. 0. 0. 1.]
 [0. 1. 0. 1. 0.]
 [1. 0. 1. 0. 0.]]


### Interpretation
The output contains only the one-hot encoded columns for `Gender` and `City`. `Age`, `Education`, and `Salary` were completely discarded because they were not explicitly mentioned.

## 8. Understanding `remainder='passthrough'`

What if you want to keep unmentioned columns intact without changing them?
Use **`remainder='passthrough'`**.

In [8]:
# Setting remainder='passthrough'
ct_pass = ColumnTransformer(
    transformers=[
        ('ohe_cat', OneHotEncoder(sparse_output=False), ['Gender', 'City'])
    ],
    remainder='passthrough'  # Keep Age, Education, Salary untouched!
)

transformed_pass = ct_pass.fit_transform(df)

print("Shape with remainder='passthrough':", transformed_pass.shape)
print("\nTransformed Content (Transformed columns FIRST, followed by passthrough columns):")
print(transformed_pass)

Shape with remainder='passthrough': (4, 8)

Transformed Content (Transformed columns FIRST, followed by passthrough columns):
[[0.0 1.0 0.0 1.0 0.0 25 'Graduate' 50000]
 [1.0 0.0 0.0 0.0 1.0 30 'Postgraduate' 70000]
 [0.0 1.0 0.0 1.0 0.0 28 'High School' 40000]
 [1.0 0.0 1.0 0.0 0.0 35 'Postgraduate' 85000]]


### Interpretation
With `remainder='passthrough'`, the unmentioned columns (`Age`, `Education`, `Salary`) are retained and appended at the end of the array.

| Setting | Unmentioned Columns |
| :--- | :--- |
| **`remainder='drop'`** *(default)* | Removed from final dataset |
| **`remainder='passthrough'`** | Kept as-is, appended at the end |

## 9. Multiple Preprocessing Techniques Together

Now let's see `ColumnTransformer` in its full power:
- **`Age` & `Salary`** $\longrightarrow$ `StandardScaler()`
- **`Education`** $\longrightarrow$ `OrdinalEncoder(categories=[...])`
- **`Gender` & `City`** $\longrightarrow$ `OneHotEncoder(sparse_output=False)`

In [9]:
from sklearn.preprocessing import StandardScaler

ct_multi = ColumnTransformer(
    transformers=[
        # 1. Scale continuous numerical features
        ('scale_num', StandardScaler(), ['Age', 'Salary']),
        
        # 2. Ordinally encode education
        ('ord_edu', OrdinalEncoder(categories=[['High School', 'Graduate', 'Postgraduate']]), ['Education']),
        
        # 3. One-hot encode nominal categories
        ('ohe_cat', OneHotEncoder(sparse_output=False), ['Gender', 'City'])
    ],
    remainder='drop'
)

# Run full multi-type transformation
transformed_multi = ct_multi.fit_transform(df)

print("Multi-Transformer Output Shape:", transformed_multi.shape)
print("\nTransformed Array (rounded to 2 decimals):")
print(np.round(transformed_multi, 2))

Multi-Transformer Output Shape: (4, 8)

Transformed Array (rounded to 2 decimals):
[[-1.24 -0.64  1.    0.    1.    0.    1.    0.  ]
 [ 0.14  0.5   2.    1.    0.    0.    0.    1.  ]
 [-0.41 -1.22  0.    0.    1.    0.    1.    0.  ]
 [ 1.51  1.36  2.    1.    0.    1.    0.    0.  ]]


### Interpretation
In a single step, three completely different preprocessing algorithms were applied to their respective columns, and the results were neatly joined together.

## 10. `SimpleImputer` + `ColumnTransformer`

Real datasets frequently have missing values (`NaN`).
We can include `SimpleImputer` inside `ColumnTransformer` to handle missing values automatically.

In [10]:
from sklearn.impute import SimpleImputer

# Create a dataset containing missing values (NaN)
df_with_nan = pd.DataFrame({
    'Age': [25.0, 30.0, np.nan, 40.0],
    'Salary': [50000.0, np.nan, 45000.0, 80000.0],
    'Gender': ['Male', 'Female', 'Female', 'Male'],
    'City': ['Delhi', 'Mumbai', 'Delhi', 'Bangalore']
})

print("--- Raw Dataset with Missing Values ---")
display(df_with_nan)

# Set up ColumnTransformer with SimpleImputer
ct_impute = ColumnTransformer(
    transformers=[
        # Fill missing values in Age and Salary using median
        ('impute_num', SimpleImputer(strategy='median'), ['Age', 'Salary']),
        
        # One-hot encode nominal strings
        ('ohe_cat', OneHotEncoder(sparse_output=False), ['Gender', 'City'])
    ],
    remainder='passthrough'
)

transformed_imputed = ct_impute.fit_transform(df_with_nan)

print("\n--- Transformed Output (No more NaNs!) ---")
print(transformed_imputed)

--- Raw Dataset with Missing Values ---


,Age,Salary,Gender,City
0,25.0,50000.0,Male,Delhi
1,30.0,NaN,Female,Mumbai
2,NaN,45000.0,Female,Delhi
3,40.0,80000.0,Male,Bangalore



--- Transformed Output (No more NaNs!) ---
[[2.5e+01 5.0e+04 0.0e+00 1.0e+00 0.0e+00 1.0e+00 0.0e+00]
 [3.0e+01 5.0e+04 1.0e+00 0.0e+00 0.0e+00 0.0e+00 1.0e+00]
 [3.0e+01 4.5e+04 1.0e+00 0.0e+00 0.0e+00 1.0e+00 0.0e+00]
 [4.0e+01 8.0e+04 0.0e+00 1.0e+00 1.0e+00 0.0e+00 0.0e+00]]


### Interpretation
Notice:
- `Age`'s missing value was filled with the median of `[25, 30, 40]` ($30.0$).
- `Salary`'s missing value was filled with the median of `[50000, 45000, 80000]` ($50000.0$).
- `Gender` and `City` were simultaneously converted to binary columns.

## 11. Train / Test Workflow & Preventing Data Leakage

Just like individual encoders and scalers, **you must never fit `ColumnTransformer` on test data**.
- Call `fit_transform(X_train)` on the training set.
- Call **only** `transform(X_test)` on the test set.

In [11]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
X = df.drop(columns=['Salary'])
y = df['Salary']

# Split data FIRST before any preprocessing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

print(f"X_train rows: {len(X_train)} | X_test rows: {len(X_test)}")

# Configure ColumnTransformer
ct_leakage_safe = ColumnTransformer(
    transformers=[
        ('scaler', StandardScaler(), ['Age']),
        ('ordinal', OrdinalEncoder(categories=[['High School', 'Graduate', 'Postgraduate']]), ['Education']),
        ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['Gender', 'City'])
    ]
)

# 1. Fit AND transform on TRAIN data
X_train_clean = ct_leakage_safe.fit_transform(X_train)

# 2. ONLY transform on TEST data (using train statistics!)
X_test_clean = ct_leakage_safe.transform(X_test)

print("\nX_train_clean shape:", X_train_clean.shape)
print("X_test_clean shape: ", X_test_clean.shape)

X_train rows: 2 | X_test rows: 2

X_train_clean shape: (2, 4)
X_test_clean shape:  (2, 4)


### Interpretation
- `fit_transform(X_train)` learned the mean and standard deviation of `Age`, and the categories of `Education`, `Gender`, and `City` from `X_train`.
- `transform(X_test)` used those exact learned values to scale and encode `X_test`.
- Data leakage was strictly prevented.

## 12. Complete Practical Example

Let's convert our multi-transformed output from Section 9 into a clear, labeled Pandas DataFrame so we can inspect every single column.

In [12]:
# Fit multi-transformer on our full original dataframe
X_processed_array = ct_multi.fit_transform(df)

# Retrieve feature names
column_names = ct_multi.get_feature_names_out()

# Create a clean DataFrame for inspection
df_processed = pd.DataFrame(
    np.round(X_processed_array, 2),
    columns=column_names
)

print("--- Final Preprocessed DataFrame ---")
display(df_processed)

--- Final Preprocessed DataFrame ---


,scale_num__Age,scale_num__Salary,ord_edu__Education,ohe_cat__Gender_Female,ohe_cat__Gender_Male,ohe_cat__City_Bangalore,ohe_cat__City_Delhi,ohe_cat__City_Mumbai
0,-1.24,-0.64,1.0,0.0,1.0,0.0,1.0,0.0
1,0.14,0.50,2.0,1.0,0.0,0.0,0.0,1.0
2,-0.41,-1.22,0.0,0.0,1.0,0.0,1.0,0.0
3,1.51,1.36,2.0,1.0,0.0,1.0,0.0,0.0


### Interpretation
Every row is now completely numerical, properly scaled, and ready for any Machine Learning algorithm.

## 13. Inspecting Columns with `get_feature_names_out()`

How do we know which column came from which transformer?
Modern Scikit-Learn provides `get_feature_names_out()` to inspect the exact column mapping.

In [13]:
# Inspect all feature names generated by ColumnTransformer
names = ct_multi.get_feature_names_out()

for idx, col_name in enumerate(names):
    print(f"Index {idx:2d} -> {col_name}")

Index  0 -> scale_num__Age
Index  1 -> scale_num__Salary
Index  2 -> ord_edu__Education
Index  3 -> ohe_cat__Gender_Female
Index  4 -> ohe_cat__Gender_Male
Index  5 -> ohe_cat__City_Bangalore
Index  6 -> ohe_cat__City_Delhi
Index  7 -> ohe_cat__City_Mumbai


### Interpretation
Notice the format:
`<transformer_name>__<original_feature_name>`
- `scale_num__Age`: Scaled age column from the `scale_num` step.
- `ord_edu__Education`: Ordinal column from `ord_edu`.
- `ohe_cat__Gender_Female`: Binary indicator for Female from `ohe_cat`.
This makes tracking features completely transparent.

## 14. Understanding the Transformed Output

Notice that our original dataset had **5 columns**, but the transformed dataset has **8 columns**.
Why did the number of columns increase?

In [14]:
print(f"Original column count:    {df.shape[1]}")
print(f"Transformed column count: {df_processed.shape[1]}")

print("\nExact breakdown of the 8 output columns:")
print("1. scale_num__Age        (1 numerical column -> 1 scaled column)")
print("2. scale_num__Salary     (1 numerical column -> 1 scaled column)")
print("3. ord_edu__Education    (1 ordinal column -> 1 encoded column)")
print("4. ohe_cat__Gender_Female (Gender binary column 1)")
print("5. ohe_cat__Gender_Male   (Gender binary column 2)")
print("6. ohe_cat__City_Bangalore (City binary column 1)")
print("7. ohe_cat__City_Delhi     (City binary column 2)")
print("8. ohe_cat__City_Mumbai    (City binary column 3)")
print("Total: 1 + 1 + 1 + 2 + 3 = 8 columns")

Original column count:    5
Transformed column count: 8

Exact breakdown of the 8 output columns:
1. scale_num__Age        (1 numerical column -> 1 scaled column)
2. scale_num__Salary     (1 numerical column -> 1 scaled column)
3. ord_edu__Education    (1 ordinal column -> 1 encoded column)
4. ohe_cat__Gender_Female (Gender binary column 1)
5. ohe_cat__Gender_Male   (Gender binary column 2)
6. ohe_cat__City_Bangalore (City binary column 1)
7. ohe_cat__City_Delhi     (City binary column 2)
8. ohe_cat__City_Mumbai    (City binary column 3)
Total: 1 + 1 + 1 + 2 + 3 = 8 columns


### Interpretation
Numerical scaling and ordinal encoding preserve the number of columns ($1 \rightarrow 1$).  
One-Hot Encoding expands each categorical feature into multiple binary columns ($1 \rightarrow K$).  
`ColumnTransformer` seamlessly handles this column expansion and joins everything in order.

## 15. `ColumnTransformer` + `Pipeline` (The Industry Standard)

In production Machine Learning, we don't just stop at preprocessing; we connect `ColumnTransformer` directly to an ML model using a **`Pipeline`**.

```text
Raw Features (X) ──► ColumnTransformer ──► Model (LinearRegression) ──► Target (Salary)
```

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# Separate features (X) and target (y)
X_features = df[['Age', 'Education', 'Gender', 'City']]
y_target = df['Salary']

# Build ColumnTransformer for the features
feature_preprocessor = ColumnTransformer(
    transformers=[
        ('scale_age', StandardScaler(), ['Age']),
        ('ord_edu', OrdinalEncoder(categories=[['High School', 'Graduate', 'Postgraduate']]), ['Education']),
        ('ohe_cat', OneHotEncoder(sparse_output=False), ['Gender', 'City'])
    ]
)

# Connect Preprocessor and Model into one complete Pipeline
full_ml_pipeline = Pipeline(
    steps=[
        ('preprocessor', feature_preprocessor),
        ('model', LinearRegression())
    ]
)

# Train the entire pipeline in ONE single line!
full_ml_pipeline.fit(X_features, y_target)
print("Pipeline fitted successfully!")

# Make predictions on raw data - no manual preprocessing required!
predictions = full_ml_pipeline.predict(X_features)

print("\n--- Pipeline Predictions ---")
for actual, pred in zip(y_target, predictions):
    print(f"Actual Salary: ${actual:,.0f} | Predicted: ${pred:,.0f}")

Pipeline fitted successfully!

--- Pipeline Predictions ---
Actual Salary: $50,000 | Predicted: $50,000
Actual Salary: $70,000 | Predicted: $70,000
Actual Salary: $40,000 | Predicted: $40,000
Actual Salary: $85,000 | Predicted: $85,000


### Interpretation
The beauty of combining `ColumnTransformer` with `Pipeline` is that:
1. **Raw data goes in, predictions come out.**
2. You never have to manually clean or transform new test data before calling `predict()`.
3. Preprocessing rules and the model stay bundled together as a single deployable object.

## 16. Final Mental Model

```text
               Raw Mixed Dataset
                       │
             [ ColumnTransformer ]
            (The Traffic Controller)
                       │
    ┌──────────────────┼──────────────────┐
    ▼                  ▼                  ▼
Numerical Lane    Ordinal Lane       Nominal Lane
 ['Age']          ['Education']      ['Gender', 'City']
    │                  │                  │
    ▼                  ▼                  ▼
StandardScaler   OrdinalEncoder     OneHotEncoder
    │                  │                  │
    └──────────────────┼──────────────────┘
                       │
                       ▼
            Combined Clean Output
                       │
                       ▼
             Machine Learning Model
```

> **The Big Takeaway:**  
> `ColumnTransformer` does not invent any new preprocessing technique. Instead, it acts as an intelligent traffic controller that routes each column to the right tool and puts all the pieces back together automatically.